# Benchmark JAX GPU — Geosteering AI Simulador Python v0.9.0+

**Sprint 3.4** — Executa o simulador Python otimizado em Colab com opção CPU/GPU.

## Instruções

1. **Runtime → Alterar tipo de runtime → GPU (T4/L4/A100)**
2. Execute todas as células em sequência
3. O notebook instala automaticamente `jax[cuda12]` ou `jax[cpu]` conforme hardware detectado

## Comparação esperada

| Plataforma | Perfil small | Perfil medium | Perfil large |
|:---|---:|---:|---:|
| Fortran i9 8T (ref) | 58.856 mod/h | — | — |
| Python Numba CPU local | 663k mod/h | 170k | 26k |
| Python Numba Colab T4 CPU | ~50k mod/h | ~15k | ~3k |
| Python JAX Colab T4 GPU | meta 200k+ | meta 60k+ | meta 10k+ |

## 1. Detecção de ambiente e instalação

In [ ]:
import subprocess
import sys

# Detecta GPU via nvidia-smi
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                            capture_output=True, text=True, timeout=5)
    HAS_GPU = result.returncode == 0
    GPU_NAME = result.stdout.strip() if HAS_GPU else 'N/A'
except (FileNotFoundError, subprocess.TimeoutExpired):
    HAS_GPU = False
    GPU_NAME = 'N/A'

print(f'GPU detectada: {HAS_GPU} ({GPU_NAME})')
print(f'Python: {sys.version.split()[0]}')

In [ ]:
# Instala dependências — escolhe jax[cuda12] se GPU, jax[cpu] caso contrário
if HAS_GPU:
    !pip install --quiet 'jax[cuda12]' numba
else:
    !pip install --quiet 'jax[cpu]' numba

# Clone do repositório (pode precisar de token para repo privado — ajuste conforme)
!git clone -q https://github.com/daniel-guitarplayer-8/geosteering-ai.git /content/geosteering-ai || echo 'repo já existe'
%cd /content/geosteering-ai
!pip install --quiet -e . 2>&1 | tail -3

## 2. Verificação do ambiente JAX

In [ ]:
import jax
import numpy as np

from geosteering_ai.simulation._jax import HAS_JAX
from geosteering_ai.simulation._numba.propagation import HAS_NUMBA

print(f'JAX versão:    {jax.__version__}')
print(f'JAX devices:   {jax.devices()}')
print(f'Backend default: {jax.default_backend()}')
print(f'HAS_JAX: {HAS_JAX}, HAS_NUMBA: {HAS_NUMBA}')
print(f'NumPy: {np.__version__}')

## 3. Benchmark — Backend Numba (CPU paralelo @njit + prange)

In [ ]:
from geosteering_ai.simulation.benchmarks.bench_forward import run_benchmark

print('### Benchmark Numba CPU (Sprint 2.9 paralelo) ###')
for size in ['small', 'medium', 'large']:
    n_iter = 5 if size != 'large' else 3
    result = run_benchmark(size=size, n_iter=n_iter, n_warmup=3)
    print(f'\n{size}: {result.throughput_models_per_hour:,.0f} mod/h '
          f'| {100*result.throughput_models_per_hour/58856:.1f}% Fortran')

## 4. Benchmark — Backend JAX (CPU ou GPU conforme runtime)

In [ ]:
import math
import time

from geosteering_ai.simulation._jax.kernel import fields_in_freqs_jax_batch
from geosteering_ai.simulation.filters import FilterLoader
from geosteering_ai.simulation.validation.canonical_models import get_canonical_model

filt = FilterLoader().load('werthmuller_201pt')

def bench_jax_profile(n_layers, n_positions, n_iter=5):
    """Roda JAX híbrido para um perfil e retorna mod/h."""
    np.random.seed(42)
    rho_h = np.exp(np.random.uniform(np.log(0.5), np.log(500), n_layers))
    rho_v = rho_h * 1.5
    esp = np.ones(n_layers - 2) * 2.0
    positions_z = np.linspace(-5, 15, n_positions)
    freqs_hz = np.array([20000.0])

    # Warmup (JIT cache)
    for _ in range(3):
        _ = fields_in_freqs_jax_batch(
            positions_z, 0.5, 0.0, 0.0,
            n_layers, rho_h, rho_v, esp, freqs_hz,
            filt.abscissas, filt.weights_j0, filt.weights_j1,
        )

    t0 = time.perf_counter()
    for _ in range(n_iter):
        _ = fields_in_freqs_jax_batch(
            positions_z, 0.5, 0.0, 0.0,
            n_layers, rho_h, rho_v, esp, freqs_hz,
            filt.abscissas, filt.weights_j0, filt.weights_j1,
        )
    t_mean = (time.perf_counter() - t0) / n_iter
    return 3600.0 / t_mean


print('### Benchmark JAX híbrido ###')
print(f'Device: {jax.default_backend()}')
print(f'\nsmall (3 cam, 100 pts):   {bench_jax_profile(3, 100, n_iter=5):,.0f} mod/h')
print(f'medium (7 cam, 300 pts):  {bench_jax_profile(7, 300, n_iter=5):,.0f} mod/h')
print(f'large (22 cam, 601 pts):  {bench_jax_profile(22, 601, n_iter=3):,.0f} mod/h')

## 5. Validação com modelos canônicos (Oklahoma, Devine, Hou, Viking Graben)

In [ ]:
from geosteering_ai.simulation.visualization import plot_all_canonical_models

paths = plot_all_canonical_models(
    output_dir='/content/canonical_plots',
    frequency_hz=20_000.0,
    tr_spacing_m=1.0,
    dip_deg=0.0,
    dpi=150,
    parallel=True,  # Sprint 2.9 @njit + prange
)
print(f'\nGerados {len(paths)} plots de modelos canônicos')
for p in paths:
    print(f'  - {p.name}')

In [ ]:
# Exibe plot de um modelo para verificação visual
from IPython.display import Image, display
display(Image(str(paths[0])))

## 6. Resumo

Este notebook executou:
1. Instalação automática de JAX (CPU ou CUDA) + Numba + pacote
2. Benchmark Numba paralelo (Sprint 2.9) nos 3 perfis
3. Benchmark JAX híbrido (Sprint 3.2 + 3.3) nos 3 perfis
4. Geração e visualização dos 7 modelos canônicos

Para obter speedup GPU máximo, execute em runtime **A100** (Colab Pro+).